# Bradley-Terry Models: VFL 1916 Example

In [1]:
import os
import sys

root_dir = os.path.dirname(os.path.abspath(os.curdir))
src_dir = os.path.join(root_dir, 'src', 'python')
sys.path.append(src_dir)
# print(sys.path[-1])

This is a brief example using our own package to fit some simple Bradley-Terry models to the [VFL 1916 data](https://en.wikipedia.org/wiki/1916_VFL_season).

In [2]:
import pandas as pd

from DynamicModel import *
from utils import *
from visualisation import *

path = "../data/VFL1916.csv"
vfl_df = pd.read_csv(path, header=0)
vfl_df["is_neutral"] = 0
vfl_df["year"] = 1916

# Make VAN data
vfl_mat = make_time_blocked_win_matrix(vfl_df.drop("is_neutral", axis=1))
vfl_mat[1916]

,Carlton,Collingwood,Fitzroy,Richmond
Carlton,0.0,3.0,3.0,4.0
Collingwood,1.0,0.0,3.5,2.0
Fitzroy,1.0,0.5,0.0,1.0
Richmond,0.0,2.0,3.0,0.0


In [3]:
# For a rough overview of the team strengths
make_victory_totals(vfl_mat[1916])

,Team,Wins,Losses,Total,Win Rate
0,Carlton,10.0,2.0,12.0,0.833333
1,Collingwood,6.5,5.5,12.0,0.541667
2,Fitzroy,2.5,9.5,12.0,0.208333
3,Richmond,5.0,7.0,12.0,0.416667


## Vanilla Bradley-Terry (VANBT)

The most standard Bradley-Terry model where each team has a 'strength' parameter.
There are no home-ground effects.

The ranking of estimated model parameters seems to align with what we expect from the win-loss ratios above.

In [4]:
van = VANBT().fit(vfl_mat)
van.rebase_abilities("worst")
van.get_ranking()

,1916,Average
Carlton,2.365038,2.365038
Collingwood,1.228527,1.228527
Richmond,0.787608,0.787608
Fitzroy,0.000000,0.000000


## Common Home-ground Advantage (HGA) Bradley-Terry (CHABT)

(Alternatively, home-field advantage / HFA)

The data requirements are different for HFA models, since we need to block wins by the venue. Our model considers neutral venues, but we will assume there are none for the VFL data. Currently, we will just need to add a dummy column of zeroes to indicate which matches are neutral (by default assuming the first column is the home team).

In [5]:
vfl_mat_venues = make_time_blocked_venue_win_matrices(vfl_df)
vfl_mat_venues[1916]["home"]

,Carlton,Collingwood,Fitzroy,Richmond
Carlton,0.0,1.0,1.0,2.0
Collingwood,0.0,0.0,2.0,1.0
Fitzroy,0.0,0.5,0.0,1.0
Richmond,0.0,1.0,2.0,0.0


In [6]:
cha = CHABT().fit(vfl_mat_venues)
cha.rebase_abilities("worst")
cha.get_ranking()

,1916,Average
Carlton,2.372262,2.372262
Collingwood,1.232391,1.232391
Richmond,0.790082,0.790082
Fitzroy,0.000000,0.000000


In [7]:
print(f"Common HGA: {cha.get_param('hga'):.4f}")

Common HGA: -0.1213


## Team-specific HGA Bradley-Terry (TSABT)

Rather than a common HGA for all teams, in this model each team has their own HGA parameter.

In [8]:
tsa = TSABT().fit(vfl_mat_venues)
tsa.rebase_abilities("worst")
tsa.get_ranking()

,1916,Average
Carlton,22.043776,22.043776
Collingwood,1.773091,1.773091
Richmond,0.374058,0.374058
Fitzroy,0.000000,0.000000


In [9]:
tsa.params[-tsa.n_teams:]

array([-20.55206859,  -0.47870105,   0.49958682,   1.8096146 ])

## Model Comparison Tests

In [10]:
from itertools import combinations
from hypothesis_tests import *

# simple_report(dyvan, dycha, N)

models = (van, cha, tsa)
for pair in list(combinations(models, 2)):
    print(pair[0].__class__.__name__, "vs", pair[1].__class__.__name__, "---", simple_report(pair[0], pair[1])["lrt"])

VANBT vs CHABT --- {'stat': np.float64(0.060590486676012745), 'df': 1, 'p_value': np.float64(0.8055650747558655)}
VANBT vs TSABT --- {'stat': np.float64(4.831507367077805), 'df': 4, 'p_value': np.float64(0.3050268367489471)}
CHABT vs TSABT --- {'stat': np.float64(4.770916880401792), 'df': 3, 'p_value': np.float64(0.18936109624630731)}


This indicates that there is no significant improvement in the CHA model over the VAN model ($p=0.8$) and hence any subsequent models.